# Setup

In [1]:
from pprint import pprint
from dotenv import dotenv_values


cfg = dotenv_values()

SEED = cfg['SEED']
CHAT_SMALL = cfg['CHAT_SMALL']
CHAT_MODEL = cfg['CHAT_MODEL']

# 1. Simple Agent

- Chat model: for conversation
- Tools: a simple tool for searching wikipedia
- Memory: a simple in-memory state, conversation history with summarization

In [18]:
from langchain_ollama import ChatOllama
from langchain.messages import SystemMessage, HumanMessage
from langchain_core.messages.utils import count_tokens_approximately
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolCallLimitMiddleware,
    ToolErrorMiddleware,
    ToolCallRequest,
    SummarizationMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver
import wikipediaapi


# 1. Chat model
chat = ChatOllama(
    model=CHAT_MODEL,
    reasoning=False,
    seed=SEED,
)

# 2. Tools
wiki = wikipediaapi.Wikipedia(
    user_agent='LangChainPrac/1.0',
    language='en',
)

@tool
def search_wiki(keyword: str) -> str:
    """Search wiki by keyword and return text."""
    pages = wiki.search(keyword, limit=2).pages

    results = []
    for keyword, value in pages.items():
        results.append(f"{value.text[:200]} ...\n")

    return results

# 3. Agent
system_prompt = SystemMessage(
    content=(
        "You MUST use provided tool to search the wiki "
        "before answer user question."
    )
)

def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    return (
        "Error for searching:\n"
        f"Tool: {request.tool_call['name']}, "
        f"Error: {type(exc).__name__}"
    )

agent = create_agent(
    model=chat,
    tools=[search_wiki],
    system_prompt=system_prompt,
    middleware=[
        ToolCallLimitMiddleware(run_limit=3),
        ToolErrorMiddleware(on_error=on_error),
        SummarizationMiddleware(
            model=ChatOllama(
                model=CHAT_MODEL,
                reasoning=False,
            ),
            trigger=('tokens', 200),
            keep=('tokens', 100),
            token_counter=count_tokens_approximately,
            trim_tokens_to_summarize=1000,
            summary_prompt=(
                "Summarize conversation history up to 150 tokens. "
                "Preserve the user's questions and shortly describe the situation. "
                "Do not invent anything not in the user prompt.\n\n"
                "{messages}"
            ),
        ),
    ],
    checkpointer=InMemorySaver(),
)

# 4. Invoke
config = {"configurable": {"thread_id": "user-1"}}

resp = agent.invoke(
    input={
        'messages': [HumanMessage("What is retrieval-augmented generation?")],
    },
    config=config,
)

for msg in resp['messages']:
    msg.pretty_print()

print("-"*80, "\n\n\n")

resp = agent.invoke(
    input={
        'messages': [HumanMessage("Which keyword did the user ask before?")],
    },
    config=config,
)

for msg in resp['messages']:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

User asks: "What is retrieval-augmented generation?"  
Situation: The user seeks a definition or explanation of retrieval-augmented generation (RAG), a technique combining information retrieval with generative AI to produce responses based on retrieved context, enhancing accuracy and reliability.
================================== Ai Message ==================================
Tool Calls:
  search_wiki (903cfa24-0fd2-4fd2-8e88-e4a0df21cd16)
 Call ID: 903cfa24-0fd2-4fd2-8e88-e4a0df21cd16
  Args:
    keyword: retrieval-augmented generation
================================= Tool Message =================================
Name: search_wiki

["Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a sp ...\n", "A vector databa

# 2. RAG

In [10]:
import wikipediaapi
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import models
from sentence_transformers import CrossEncoder
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore


# 1. Documents
wiki = wikipediaapi.Wikipedia(
    user_agent='LangChainRAG/1.0',
    language='en',
)
pages = wiki.search('Retrieval-augmented generation', limit=3).pages
docs = [
    Document(
        page_content=page.text,
        metadata={'title': title},
    )
    for title, page in pages.items()
]

# 2. Split & Embedding
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
)
chunks = splitter.split_documents(docs)

emb_dense = OllamaEmbeddings(model='qwen3-embedding:0.6b')
emb_sparse = FastEmbedSparse('Qdrant/bm25')

# 3. Store & Vector DB
vector_db = QdrantVectorStore.from_documents(   # .from_existing_collection()
    collection_name='wiki_hybrid',
    # location=':memory:',
    path='../../tmp/rag_prac',
    documents=chunks,
    embedding=emb_dense,
    sparse_embedding=emb_sparse,
    sparse_vector_params={'modifier': models.Modifier.IDF},
    retrieval_mode=RetrievalMode.HYBRID,
)

# 4. Retriever & Reranker
retriever = vector_db.as_retriever(search_kwargs={"k": 5})
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

@tool
def search_context(query: str) -> str | None:
    """Use user query to search related documents."""
    retrieved_docs = retriever.invoke(query)
    if not retrieved_docs:
        return "No documents found."

    ranked = reranker.rank(
        query="What is RAG?",
        documents=[doc.page_content for doc in retrieved_docs],
        top_k=2,
    )
    reranked_docs = [retrieved_docs[item['corpus_id']] for item in ranked]
    context = "\n\n".join(
        f"Title: {doc.metadata['title']}\n{doc.page_content}"
        for doc in reranked_docs
    )

    return context

# 5. Agent
system_prompt = SystemMessage(
    content=(
        "You must use provided tool to search the information, "
        "and answer the user question shortly."
    )
)

agent = create_agent(
    model=ChatOllama(
        model=CHAT_MODEL,
        reasoning=False,
    ),
    tools=[search_context],
    system_prompt=system_prompt,
)

# 6. Invoke
resp = agent.invoke(
    input={
        'messages': [HumanMessage("Search and answer. What is RAG?")],
    }
)

for msg in resp['messages']:
    msg.pretty_print()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2127.10it/s]


================================ Human Message =================================

Search and answer. What is RAG?
================================== Ai Message ==================================
Tool Calls:
  search_context (3658be91-2439-4725-8298-6e24933fdb71)
 Call ID: 3658be91-2439-4725-8298-6e24933fdb71
  Args:
    query: What is RAG?
================================= Tool Message =================================
Name: search_context

Title: Retrieval-augmented generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal

Tit

# 3. Memory

In [ ]:
from pydantic import BaseModel
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.messages import SystemMessage, HumanMessage
from langchain.agents import create_agent
from langmem import (
    create_memory_store_manager,
    create_search_memory_tool,
)
import sqlite3
from langgraph.store.sqlite import SqliteStore


# 1. Memory schema
class Fact(BaseModel):
    """One specific fact."""
    content: str

# 2. Store
conn = sqlite3.connect(
    "../../tmp/memory_prac.sqlite",
    check_same_thread=False,
    isolation_level=None,
)
store = SqliteStore(
    conn=conn,
    index={
        'embed': OllamaEmbeddings(model='qwen3-embedding:0.6b'),
        'dims': 1024,
    }
)
store.setup()

# 3. Memory manager
namespace = ('memory_prac', 'user-1', 'fact')

memory_manager = create_memory_store_manager(
    ChatOllama(model=CHAT_MODEL, reasoning=False),
    schemas=[Fact],
    instructions=(
        "Call the Fact tool for each explicit fact including their name."
    ),
    enable_inserts=True,
    enable_deletes=False,
    query_limit=3,
    store=store,
    namespace=namespace,
)

# 4. Agent
search_tool = create_search_memory_tool(
    store=store,
    namespace=namespace,
    instructions="Use this tool to retrieve user memory.",
)

agent = create_agent(
    model=ChatOllama(model=CHAT_MODEL, reasoning=False),
    tools=[search_tool],
    system_prompt=SystemMessage(
        content=(
            "You must use provided tool to retrieve memory before answer user question."
        )
    ),
)

# 5. Invoke
history = "My name is yana."
memory_manager.invoke(
    input={'messages': [HumanMessage(history)]}
)

for memory in store.search(namespace, query=history, limit=3):
    print(memory.value)

resp = agent.invoke(
    input={'messages': [HumanMessage("Use provided tool to search my name.")]}
)

for msg in resp['messages']:
    msg.pretty_print()

{'kind': 'Fact', 'content': {'content': 'My name is yana.'}}
{'kind': 'Fact', 'content': {'content': 'My name is yana.'}}
================================ Human Message =================================

Use provided tool to search my name.
================================== Ai Message ==================================
Tool Calls:
  search_memory (5fe14cb9-385f-4a8d-8200-ef8bac4f3163)
 Call ID: 5fe14cb9-385f-4a8d-8200-ef8bac4f3163
  Args:
    query: my name
================================= Tool Message =================================
Name: search_memory

[{"namespace":["memory_prac","user-1","fact"],"key":"4a562199-d6d1-4871-9dc4-003e1bf4bc04","value":{"kind":"Fact","content":{"content":"My name is yana."}},"created_at":"2026-11-10T07:57:12","updated_at":"2026-11-10T07:57:12","score":0.750139981508255},{"namespace":["memory_prac","user-1","fact"],"key":"7cbb44b0-b582-4c94-9013-3779233937cf","value":{"kind":"Fact","content":{"content":"My name is yana."}},"created_at":"2026-11-10T08

# 4. Graph

# 5. GraphRAG